# Team Features

Compile all feature engineering into a model-ready dataframe. 

In [1]:
SEASON = 2025

### Previous Tournament Results

In [2]:
import pandas as pd

pd.set_option('display.max_columns', 100)

df = pd.read_parquet(r'..\data\preprocessed\mens_kaggle\tournament_results.parquet')

df = df.loc[(~df['Season'].isin([2020])) & (df['Season'] < SEASON), :].reset_index(drop=True)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results
0,2012,1101,Abilene Chr,-1.0,-1.0
1,2012,1102,Air Force,-1.0,-1.0
2,2012,1103,Akron,0.0,-0.5
3,2012,1104,Alabama,-1.0,-1.0
4,2012,1105,Alabama A&M,-1.0,-1.0
...,...,...,...,...,...
4555,2024,1476,Stonehill,-1.0,-1.0
4556,2024,1477,East Texas A&M,-1.0,-1.0
4557,2024,1478,Le Moyne,-1.0,-1.0
4558,2024,1479,Mercyhurst,-1.0,-1.0


### Barttorvik Ratings

In [3]:
df_barttorvik = pd.read_parquet(r'..\data\preprocessed\mens_barttorvik\barttorvik.csv')

df_barttorvik

,Season,TEAM,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS
0,2012,Kentucky,119.7,88.5,31.2,0.970,0.941176,53.4,41.6,40.0,25.6,17.2,18.1,38.4,31.0,65.9,38.8,32.1,21.4,8.1,48.4,48.6,27.3,30.0,66.1,81.480,0.776,95.267,72.0,27.605
1,2012,Ohio St.,115.5,85.5,30.0,0.970,0.794118,52.5,46.3,37.0,28.6,17.4,22.5,35.7,24.9,66.7,45.3,32.2,8.8,6.1,54.3,47.2,26.1,35.5,68.1,80.091,1.254,87.821,69.8,31.751
2,2012,Kansas,114.7,88.1,26.6,0.954,0.818182,54.0,43.9,41.1,34.3,19.6,20.7,34.9,28.6,67.5,40.0,34.7,15.1,8.2,58.6,52.4,30.5,32.2,67.9,81.633,2.178,72.255,69.6,30.956
3,2012,Michigan St.,112.7,86.7,26.0,0.953,0.794118,52.7,43.0,39.0,34.2,19.8,19.7,37.2,27.5,65.3,42.6,29.2,12.7,7.3,59.1,52.2,27.8,35.6,66.1,80.061,1.672,67.935,69.8,34.349
4,2012,North Carolina,115.8,89.6,26.2,0.950,0.852941,50.0,45.1,38.1,22.0,16.2,18.5,40.5,27.7,72.8,43.5,31.9,13.8,6.9,58.1,46.4,22.9,35.9,72.7,82.601,1.384,91.131,68.2,27.923
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4225,2024,Stonehill,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832
4226,2024,Saint Francis,93.3,117.3,-24.0,0.067,0.214286,47.2,53.0,32.6,35.4,21.2,17.1,32.9,31.3,66.4,52.9,35.4,10.1,11.8,52.6,52.0,35.1,37.1,66.4,79.692,0.327,0.200,60.6,8.567
4227,2024,IU Indy,92.7,116.9,-24.2,0.065,0.103448,46.5,58.2,33.2,33.4,21.3,18.5,30.0,35.5,68.6,59.0,38.0,6.0,7.7,42.4,55.3,24.3,37.5,68.3,79.492,2.044,4.750,72.3,10.790
4228,2024,Coppin St.,84.7,110.0,-25.3,0.047,0.068966,42.1,51.3,31.1,38.3,22.9,21.8,27.0,38.6,67.3,51.0,34.5,8.0,9.5,40.7,58.6,34.4,37.6,67.3,80.172,1.292,4.358,72.6,9.758


In [4]:
df_spellings = pd.read_csv(
    r'..\data\unprocessed\kaggle\MTeamSpellings.csv', 
    encoding='cp1252'  # fixes issue with fancy quotes
)

df_spellings.loc[df_spellings.shape[0]] = ['fdu', 1192]

df_spellings

,TeamNameSpelling,TeamID
0,a&m-corpus chris,1394
1,a&m-corpus christi,1394
2,abilene chr,1101
3,abilene christian,1101
4,abilene-christian,1101
...,...,...
1173,youngstown st.,1464
1174,youngstown state,1464
1175,youngstown-st,1464
1176,youngstown-state,1464


In [5]:
spelling_to_id = dict(zip(df_spellings['TeamNameSpelling'], df_spellings['TeamID']))

len(spelling_to_id)

1178

In [6]:
from fuzzywuzzy.fuzz import token_sort_ratio
from fuzzywuzzy import process
from tqdm.autonotebook import tqdm

def match_names(team_spellings, new_data_teams):
    df_match = pd.DataFrame(
        [
            [
                new_data_team,
                *process.extract(
                    new_data_team,
                    team_spellings,
                    scorer=token_sort_ratio,
                    limit=1
                )[0][:2]
            ] for new_data_team in tqdm(new_data_teams)
        ],
        columns=['New Data Team', 'Team Spelling', 'Match Score']
    ).sort_values('Match Score', ignore_index=True)

    team_to_spelling = dict(zip(df_match['New Data Team'], df_match['Team Spelling']))

    return df_match, team_to_spelling

C:\Users\mhugh\AppData\Local\Temp\ipykernel_12048\2578028529.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [7]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_barttorvik['TEAM'].unique())

df_match.head(25)

  0%|          | 0/365 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Queens,Queens (NC),80
1,UT Rio Grande Valley,texas rio grande valley,88
2,Saint Francis,saint francis (ny),90
3,Texas A&M Commerce,tx a&m commerce,91
4,Cal St. Bakersfield,cal state bakersfield,92
5,Mississippi Valley St.,mississippi valley state,93
6,Southeast Missouri St.,southeast missouri state,93
7,Texas A&M Corpus Chris,texas a&m-corpus christi,96
8,Bethune Cookman,bethune-cookman,100
9,Morehead St.,morehead st,100


In [8]:
df_barttorvik.insert(1, 'TeamID', df_barttorvik['TEAM'].map(team_to_spelling).map(spelling_to_id))

df_barttorvik

,Season,TeamID,TEAM,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS
0,2012,1246,Kentucky,119.7,88.5,31.2,0.970,0.941176,53.4,41.6,40.0,25.6,17.2,18.1,38.4,31.0,65.9,38.8,32.1,21.4,8.1,48.4,48.6,27.3,30.0,66.1,81.480,0.776,95.267,72.0,27.605
1,2012,1326,Ohio St.,115.5,85.5,30.0,0.970,0.794118,52.5,46.3,37.0,28.6,17.4,22.5,35.7,24.9,66.7,45.3,32.2,8.8,6.1,54.3,47.2,26.1,35.5,68.1,80.091,1.254,87.821,69.8,31.751
2,2012,1242,Kansas,114.7,88.1,26.6,0.954,0.818182,54.0,43.9,41.1,34.3,19.6,20.7,34.9,28.6,67.5,40.0,34.7,15.1,8.2,58.6,52.4,30.5,32.2,67.9,81.633,2.178,72.255,69.6,30.956
3,2012,1277,Michigan St.,112.7,86.7,26.0,0.953,0.794118,52.7,43.0,39.0,34.2,19.8,19.7,37.2,27.5,65.3,42.6,29.2,12.7,7.3,59.1,52.2,27.8,35.6,66.1,80.061,1.672,67.935,69.8,34.349
4,2012,1314,North Carolina,115.8,89.6,26.2,0.950,0.852941,50.0,45.1,38.1,22.0,16.2,18.5,40.5,27.7,72.8,43.5,31.9,13.8,6.9,58.1,46.4,22.9,35.9,72.7,82.601,1.384,91.131,68.2,27.923
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4225,2024,1476,Stonehill,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832
4226,2024,1383,Saint Francis,93.3,117.3,-24.0,0.067,0.214286,47.2,53.0,32.6,35.4,21.2,17.1,32.9,31.3,66.4,52.9,35.4,10.1,11.8,52.6,52.0,35.1,37.1,66.4,79.692,0.327,0.200,60.6,8.567
4227,2024,1237,IU Indy,92.7,116.9,-24.2,0.065,0.103448,46.5,58.2,33.2,33.4,21.3,18.5,30.0,35.5,68.6,59.0,38.0,6.0,7.7,42.4,55.3,24.3,37.5,68.3,79.492,2.044,4.750,72.3,10.790
4228,2024,1164,Coppin St.,84.7,110.0,-25.3,0.047,0.068966,42.1,51.3,31.1,38.3,22.9,21.8,27.0,38.6,67.3,51.0,34.5,8.0,9.5,40.7,58.6,34.4,37.6,67.3,80.172,1.292,4.358,72.6,9.758


In [9]:
df_barttorvik.loc[df_barttorvik['TeamID'].isna(), :]

,Season,TeamID,TEAM,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS


In [10]:
df = pd.merge(
    df,
    df_barttorvik.drop(columns=['TEAM']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,1102,Air Force,-1.0,-1.0,98.5,100.0,-1.5,0.457,0.407407,51.1,48.4,39.4,38.2,20.6,21.5,19.8,31.5,62.7,48.4,32.3,10.0,9.4,64.0,60.6,40.0,41.6,62.3,79.876,1.607,0.766,68.2,21.142
2,2012,1103,Akron,0.0,-0.5,105.1,96.9,8.2,0.718,0.636364,51.5,46.4,40.0,34.0,21.0,20.6,34.8,33.2,67.5,47.3,29.6,10.6,9.0,55.1,52.3,29.6,29.5,67.8,80.958,1.776,24.205,69.2,19.260
3,2012,1104,Alabama,-1.0,-1.0,105.0,88.1,16.9,0.883,0.656250,49.0,43.4,36.5,38.6,20.4,21.4,33.9,31.9,63.1,43.9,28.3,11.9,9.4,52.0,51.1,27.1,32.9,63.1,79.579,1.010,84.233,71.5,26.959
4,2012,1105,Alabama A&M,-1.0,-1.0,88.4,111.1,-22.7,0.067,0.192308,45.1,49.2,35.5,52.9,23.9,20.6,30.4,33.6,67.6,48.0,35.1,10.3,11.3,48.9,45.9,30.9,25.4,67.9,79.249,1.561,2.167,65.2,8.917
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4566,2024,1476,Stonehill,-1.0,-1.0,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832
4567,2024,1477,East Texas A&M,-1.0,-1.0,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953
4568,2024,1478,Le Moyne,-1.0,-1.0,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904
4569,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
df.loc[df['WIN%'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2012,1109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2012,1118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2012,1121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2012,1128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4522,2024,1432,Utica,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4535,2024,1445,W Salem St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4536,2024,1446,W Texas A&M,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4569,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Barttorvik Previous Seasons

In [12]:
df_barttorvik_prev = pd.read_parquet(r'..\data\preprocessed\mens_barttorvik_full_season\barttorvik_full_season.parquet')

df_barttorvik_prev

,Season,TEAM,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
0,2012,Abilene Christian,NaN,NaN,NaN,NaN
1,2012,Air Force,0.578,2.881,0.462750,-1.377750
2,2012,Akron,0.605,3.741,0.677500,6.696000
3,2012,Alabama,0.842,14.255,0.780750,11.435000
4,2012,Alabama A&M,0.128,-16.046,0.103500,-18.409250
...,...,...,...,...,...,...
5133,2025,Wright St.,0.553,2.124,0.551750,2.084500
5134,2025,Wyoming,0.530,1.125,0.590750,3.550500
5135,2025,Xavier,0.800,12.669,0.825750,14.518500
5136,2025,Yale,0.723,8.782,0.673667,6.763333


In [13]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_barttorvik_prev['TEAM'].unique())

df_match.head(25)

  0%|          | 0/367 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Queens,Queens (NC),80
1,UT Rio Grande Valley,texas rio grande valley,88
2,Saint Francis,saint francis (ny),90
3,Texas A&M Commerce,tx a&m commerce,91
4,Winston Salem St.,winston-salem-state,91
5,Cal St. Bakersfield,cal state bakersfield,92
6,Southeast Missouri St.,southeast missouri state,93
7,Mississippi Valley St.,mississippi valley state,93
8,Texas A&M Corpus Chris,texas a&m-corpus christi,96
9,Rice,rice,100


In [14]:
df_barttorvik_prev.insert(1, 'TeamID', df_barttorvik_prev['TEAM'].map(team_to_spelling).map(spelling_to_id))

df_barttorvik_prev

,Season,TeamID,TEAM,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
0,2012,1101,Abilene Christian,NaN,NaN,NaN,NaN
1,2012,1102,Air Force,0.578,2.881,0.462750,-1.377750
2,2012,1103,Akron,0.605,3.741,0.677500,6.696000
3,2012,1104,Alabama,0.842,14.255,0.780750,11.435000
4,2012,1105,Alabama A&M,0.128,-16.046,0.103500,-18.409250
...,...,...,...,...,...,...,...
5133,2025,1460,Wright St.,0.553,2.124,0.551750,2.084500
5134,2025,1461,Wyoming,0.530,1.125,0.590750,3.550500
5135,2025,1462,Xavier,0.800,12.669,0.825750,14.518500
5136,2025,1463,Yale,0.723,8.782,0.673667,6.763333


In [15]:
df = pd.merge(
    df,
    df_barttorvik_prev.drop(columns=['TEAM']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,1102,Air Force,-1.0,-1.0,98.5,100.0,-1.5,0.457,0.407407,51.1,48.4,39.4,38.2,20.6,21.5,19.8,31.5,62.7,48.4,32.3,10.0,9.4,64.0,60.6,40.0,41.6,62.3,79.876,1.607,0.766,68.2,21.142,0.578,2.881,0.46275,-1.37775
2,2012,1103,Akron,0.0,-0.5,105.1,96.9,8.2,0.718,0.636364,51.5,46.4,40.0,34.0,21.0,20.6,34.8,33.2,67.5,47.3,29.6,10.6,9.0,55.1,52.3,29.6,29.5,67.8,80.958,1.776,24.205,69.2,19.260,0.605,3.741,0.67750,6.69600
3,2012,1104,Alabama,-1.0,-1.0,105.0,88.1,16.9,0.883,0.656250,49.0,43.4,36.5,38.6,20.4,21.4,33.9,31.9,63.1,43.9,28.3,11.9,9.4,52.0,51.1,27.1,32.9,63.1,79.579,1.010,84.233,71.5,26.959,0.842,14.255,0.78075,11.43500
4,2012,1105,Alabama A&M,-1.0,-1.0,88.4,111.1,-22.7,0.067,0.192308,45.1,49.2,35.5,52.9,23.9,20.6,30.4,33.6,67.6,48.0,35.1,10.3,11.3,48.9,45.9,30.9,25.4,67.9,79.249,1.561,2.167,65.2,8.917,0.128,-16.046,0.10350,-18.40925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4589,2024,1476,Stonehill,-1.0,-1.0,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN
4590,2024,1477,East Texas A&M,-1.0,-1.0,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN
4591,2024,1478,Le Moyne,-1.0,-1.0,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN
4592,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
df.loc[df['Past Year BARTHAG'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2012,1109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2012,1118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2012,1121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2012,1128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4558,2024,1445,W Salem St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4559,2024,1446,W Texas A&M,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4591,2024,1478,Le Moyne,-1.0,-1.0,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.2,76.5,7.904,NaN,NaN,NaN,NaN
4592,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
df.loc[df['Past Year BARTHAG'].isna() & (df['Past 4 Years Tournament Results'] > -1.0), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
3681,2022,1335,Penn,-1.0,-0.666667,104.5,108.8,-4.3,0.386,0.428571,51.3,52.1,24.1,28.8,17.0,15.3,25.4,28.0,69.0,52.1,34.7,6.1,9.0,46.9,44.2,40.2,38.1,68.9,78.818,1.479,2.690,72.8,17.117,NaN,NaN,0.647333,5.376667
3812,2022,1463,Yale,-1.0,-0.666667,99.8,98.4,1.4,0.541,0.620690,50.1,48.9,32.7,31.1,18.3,18.2,25.6,26.5,70.3,51.1,30.4,9.2,9.7,46.4,48.0,36.3,41.2,69.8,78.217,1.898,9.416,73.7,14.644,NaN,NaN,0.628000,5.128333


### My Rankings

In [18]:
df_rankings = pd.concat(
    (
        pd.read_parquet(fr'..\data\preprocessed\mens_my_rankings\my_rankings_{season}.parquet')
        .assign(Season=season)
        for season in range(2012, SEASON) if season != 2020
    ),
    ignore_index=True
)

df_rankings.insert(0, 'Season', df_rankings.pop('Season'))

df_rankings.drop(columns=['Strength'], inplace=True)

df_rankings

,Season,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2012,Kentucky,3.565314,0.297701,1.166637,0.868936,66.950567
1,2012,North Carolina,3.219256,0.242387,1.124366,0.881978,73.342503
2,2012,Syracuse,3.027328,0.247189,1.132119,0.884929,67.141049
3,2012,Ohio State,2.922136,0.289967,1.133910,0.843942,68.205218
4,2012,Duke,2.910209,0.197090,1.147153,0.950063,68.353372
...,...,...,...,...,...,...,...
4224,2024,Virginia Military Institute,-2.050009,-0.226840,0.888982,1.115822,75.056385
4225,2024,Stonehill,-2.164153,-0.214380,0.914280,1.128660,69.240111
4226,2024,Coppin State,-2.203441,-0.250565,0.847038,1.097603,67.665417
4227,2024,Mississippi Valley State,-2.690286,-0.330596,0.847257,1.177853,65.666144


In [19]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_rankings['Team'].unique())

df_match.head(25)

  0%|          | 0/365 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Hartford Hawks,hartford,73
1,St. Francis (NY) Terriers,st francis (ny),74
2,Savannah State Tigers,savannah state,80
3,Texas A&M-Commerce,tx a&m commerce,91
4,Kentucky,kentucky,100
5,Eastern Kentucky,eastern kentucky,100
6,North Dakota,north dakota,100
7,Delaware State,delaware state,100
8,Marist,marist,100
9,Stetson,stetson,100


In [20]:
df_rankings.insert(1, 'TeamID', df_rankings['Team'].map(team_to_spelling).map(spelling_to_id))

df_rankings

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2012,1246,Kentucky,3.565314,0.297701,1.166637,0.868936,66.950567
1,2012,1314,North Carolina,3.219256,0.242387,1.124366,0.881978,73.342503
2,2012,1393,Syracuse,3.027328,0.247189,1.132119,0.884929,67.141049
3,2012,1326,Ohio State,2.922136,0.289967,1.133910,0.843942,68.205218
4,2012,1181,Duke,2.910209,0.197090,1.147153,0.950063,68.353372
...,...,...,...,...,...,...,...,...
4224,2024,1440,Virginia Military Institute,-2.050009,-0.226840,0.888982,1.115822,75.056385
4225,2024,1476,Stonehill,-2.164153,-0.214380,0.914280,1.128660,69.240111
4226,2024,1164,Coppin State,-2.203441,-0.250565,0.847038,1.097603,67.665417
4227,2024,1290,Mississippi Valley State,-2.690286,-0.330596,0.847257,1.177853,65.666144


In [21]:
df_rankings.loc[df_rankings['TeamID'].isna(), :]

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo


In [22]:
df = pd.merge(
    df,
    df_rankings.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,1102,Air Force,-1.0,-1.0,98.5,100.0,-1.5,0.457,0.407407,51.1,48.4,39.4,38.2,20.6,21.5,19.8,31.5,62.7,48.4,32.3,10.0,9.4,64.0,60.6,40.0,41.6,62.3,79.876,1.607,0.766,68.2,21.142,0.578,2.881,0.46275,-1.37775,0.155360,-0.020097,0.979952,1.000049,62.836286
2,2012,1103,Akron,0.0,-0.5,105.1,96.9,8.2,0.718,0.636364,51.5,46.4,40.0,34.0,21.0,20.6,34.8,33.2,67.5,47.3,29.6,10.6,9.0,55.1,52.3,29.6,29.5,67.8,80.958,1.776,24.205,69.2,19.260,0.605,3.741,0.67750,6.69600,1.338710,0.095748,1.045688,0.949940,68.483190
3,2012,1104,Alabama,-1.0,-1.0,105.0,88.1,16.9,0.883,0.656250,49.0,43.4,36.5,38.6,20.4,21.4,33.9,31.9,63.1,43.9,28.3,11.9,9.4,52.0,51.1,27.1,32.9,63.1,79.579,1.010,84.233,71.5,26.959,0.842,14.255,0.78075,11.43500,1.574844,0.154447,1.034203,0.879756,63.968330
4,2012,1105,Alabama A&M,-1.0,-1.0,88.4,111.1,-22.7,0.067,0.192308,45.1,49.2,35.5,52.9,23.9,20.6,30.4,33.6,67.6,48.0,35.1,10.3,11.3,48.9,45.9,30.9,25.4,67.9,79.249,1.561,2.167,65.2,8.917,0.128,-16.046,0.10350,-18.40925,-3.145351,-0.191359,0.890988,1.082347,68.157167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4589,2024,1476,Stonehill,-1.0,-1.0,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.164153,-0.214380,0.914280,1.128660,69.240111
4590,2024,1477,East Texas A&M,-1.0,-1.0,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.196470,-0.161715,0.935543,1.097258,67.458552
4591,2024,1478,Le Moyne,-1.0,-1.0,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.228214,-0.091264,1.001228,1.092492,68.446622
4592,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
df.loc[df['Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2012,1109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2012,1118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2012,1121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2012,1128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4545,2024,1432,Utica,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4558,2024,1445,W Salem St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4559,2024,1446,W Texas A&M,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4592,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
df.loc[(df['Rating'].isna()) & (df['Past 4 Years Tournament Results'] > -1.0), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
3298,2021,1335,Penn,NaN,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.600,3.591,0.61375,4.14700,NaN,NaN,NaN,NaN,NaN
3306,2021,1343,Princeton,NaN,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.572,2.671,0.58175,3.33725,NaN,NaN,NaN,NaN,NaN
3429,2021,1463,Yale,NaN,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.755,10.046,0.59975,3.98500,NaN,NaN,NaN,NaN,NaN
4328,2024,1216,Hartford,-1.0,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.042,-28.007,0.26400,-11.57475,NaN,NaN,NaN,NaN,NaN


### Starters

In [25]:
df_starters = pd.concat(
    (
        pd.read_parquet(fr'..\data\preprocessed\mens_starters\starters_{season}.parquet')
        .assign(Season=season)
        for season in range(2012, SEASON) if season != 2020
    ),
    ignore_index=True
)

df_starters.insert(0, 'Season', df_starters.pop('Season'))

df_starters.rename(columns={'Rating': 'Starters'}, inplace=True)

df_starters

,Season,Team,Starters
0,2012,Murray State,0.583076
1,2012,Michigan State,0.561808
2,2012,Duke,0.537206
3,2012,Wichita State,0.536827
4,2012,Kentucky,0.534842
...,...,...,...
4224,2024,Stonehill,-0.400817
4225,2024,Virginia Military Institute,-0.406939
4226,2024,Buffalo,-0.422366
4227,2024,Mississippi Valley State,-0.442076


In [26]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_starters['Team'].unique())

df_match.head(25)

  0%|          | 0/365 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Texas A&M-Commerce,tx a&m commerce,91
1,Murray State,murray state,100
2,Lipscomb,lipscomb,100
3,Louisiana,louisiana,100
4,Texas State,texas state,100
5,UC Irvine,uc irvine,100
6,Southern,southern,100
7,Appalachian State,appalachian state,100
8,Southeastern Louisiana,southeastern louisiana,100
9,Morgan State,morgan state,100


In [27]:
df_starters.insert(1, 'TeamID', df_starters['Team'].map(team_to_spelling).map(spelling_to_id))

df_starters

,Season,TeamID,Team,Starters
0,2012,1293,Murray State,0.583076
1,2012,1277,Michigan State,0.561808
2,2012,1181,Duke,0.537206
3,2012,1455,Wichita State,0.536827
4,2012,1246,Kentucky,0.534842
...,...,...,...,...
4224,2024,1476,Stonehill,-0.400817
4225,2024,1440,Virginia Military Institute,-0.406939
4226,2024,1138,Buffalo,-0.422366
4227,2024,1290,Mississippi Valley State,-0.442076


In [28]:
df_starters.loc[df_starters['TeamID'].isna(), :]

,Season,TeamID,Team,Starters


In [29]:
df = pd.merge(
    df,
    df_starters.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,1102,Air Force,-1.0,-1.0,98.5,100.0,-1.5,0.457,0.407407,51.1,48.4,39.4,38.2,20.6,21.5,19.8,31.5,62.7,48.4,32.3,10.0,9.4,64.0,60.6,40.0,41.6,62.3,79.876,1.607,0.766,68.2,21.142,0.578,2.881,0.46275,-1.37775,0.155360,-0.020097,0.979952,1.000049,62.836286,-0.066193
2,2012,1103,Akron,0.0,-0.5,105.1,96.9,8.2,0.718,0.636364,51.5,46.4,40.0,34.0,21.0,20.6,34.8,33.2,67.5,47.3,29.6,10.6,9.0,55.1,52.3,29.6,29.5,67.8,80.958,1.776,24.205,69.2,19.260,0.605,3.741,0.67750,6.69600,1.338710,0.095748,1.045688,0.949940,68.483190,0.359659
3,2012,1104,Alabama,-1.0,-1.0,105.0,88.1,16.9,0.883,0.656250,49.0,43.4,36.5,38.6,20.4,21.4,33.9,31.9,63.1,43.9,28.3,11.9,9.4,52.0,51.1,27.1,32.9,63.1,79.579,1.010,84.233,71.5,26.959,0.842,14.255,0.78075,11.43500,1.574844,0.154447,1.034203,0.879756,63.968330,0.226809
4,2012,1105,Alabama A&M,-1.0,-1.0,88.4,111.1,-22.7,0.067,0.192308,45.1,49.2,35.5,52.9,23.9,20.6,30.4,33.6,67.6,48.0,35.1,10.3,11.3,48.9,45.9,30.9,25.4,67.9,79.249,1.561,2.167,65.2,8.917,0.128,-16.046,0.10350,-18.40925,-3.145351,-0.191359,0.890988,1.082347,68.157167,-0.315889
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4589,2024,1476,Stonehill,-1.0,-1.0,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.164153,-0.214380,0.914280,1.128660,69.240111,-0.400817
4590,2024,1477,East Texas A&M,-1.0,-1.0,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.196470,-0.161715,0.935543,1.097258,67.458552,-0.111816
4591,2024,1478,Le Moyne,-1.0,-1.0,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.228214,-0.091264,1.001228,1.092492,68.446622,-0.070744
4592,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
df.loc[df['Starters'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2012,1109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2012,1118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2012,1121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2012,1128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4545,2024,1432,Utica,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4558,2024,1445,W Salem St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4559,2024,1446,W Texas A&M,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4592,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
df.loc[(df['Starters'].isna()) & (df['Past 4 Years Tournament Results'] > -1.0), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters
3298,2021,1335,Penn,NaN,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.600,3.591,0.61375,4.14700,NaN,NaN,NaN,NaN,NaN,NaN
3306,2021,1343,Princeton,NaN,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.572,2.671,0.58175,3.33725,NaN,NaN,NaN,NaN,NaN,NaN
3429,2021,1463,Yale,NaN,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.755,10.046,0.59975,3.98500,NaN,NaN,NaN,NaN,NaN,NaN
4328,2024,1216,Hartford,-1.0,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.042,-28.007,0.26400,-11.57475,NaN,NaN,NaN,NaN,NaN,NaN


### Openskill Ratings

In [32]:
df_os = pd.concat(
    (
        pd.read_parquet(fr'..\data\preprocessed\mens_os_rankings\os_rankings_{season}.parquet')
        .assign(Season=season)
        for season in range(2012, SEASON) if season != 2020
    ),
    ignore_index=True
)

df_os.insert(0, 'Season', df_os.pop('Season'))

df_os.drop(columns=['Sigma'], inplace=True)

df_os

,Season,Team,Mu,OS Rating
0,2012,Kentucky,52.874948,39.460082
1,2012,Syracuse,52.172554,38.563683
2,2012,Michigan State,49.305377,37.216686
3,2012,Missouri,49.958466,37.028387
4,2012,North Carolina,48.170494,35.567693
...,...,...,...,...
4224,2024,IUPUI,1.774340,-12.158040
4225,2024,Virginia Military Institute,2.390031,-12.570655
4226,2024,Detroit Mercy,1.112006,-13.481287
4227,2024,Coppin State,0.370521,-13.662552


In [33]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_os['Team'].unique())

df_match.head(25)

  0%|          | 0/365 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Hartford Hawks,hartford,73
1,St. Francis (NY) Terriers,st francis (ny),74
2,Savannah State Tigers,savannah state,80
3,Texas A&M-Commerce,tx a&m commerce,91
4,Rhode Island,rhode island,100
5,UC Irvine,uc irvine,100
6,Hofstra,hofstra,100
7,Ball State,ball state,100
8,UNC Greensboro,unc greensboro,100
9,Texas Tech,texas tech,100


In [34]:
df_os.insert(1, 'TeamID', df_os['Team'].map(team_to_spelling).map(spelling_to_id))

df_os

,Season,TeamID,Team,Mu,OS Rating
0,2012,1246,Kentucky,52.874948,39.460082
1,2012,1393,Syracuse,52.172554,38.563683
2,2012,1277,Michigan State,49.305377,37.216686
3,2012,1281,Missouri,49.958466,37.028387
4,2012,1314,North Carolina,48.170494,35.567693
...,...,...,...,...,...
4224,2024,1237,IUPUI,1.774340,-12.158040
4225,2024,1440,Virginia Military Institute,2.390031,-12.570655
4226,2024,1178,Detroit Mercy,1.112006,-13.481287
4227,2024,1164,Coppin State,0.370521,-13.662552


In [35]:
df = pd.merge(
    df,
    df_os.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,1102,Air Force,-1.0,-1.0,98.5,100.0,-1.5,0.457,0.407407,51.1,48.4,39.4,38.2,20.6,21.5,19.8,31.5,62.7,48.4,32.3,10.0,9.4,64.0,60.6,40.0,41.6,62.3,79.876,1.607,0.766,68.2,21.142,0.578,2.881,0.46275,-1.37775,0.155360,-0.020097,0.979952,1.000049,62.836286,-0.066193,24.057973,10.314879
2,2012,1103,Akron,0.0,-0.5,105.1,96.9,8.2,0.718,0.636364,51.5,46.4,40.0,34.0,21.0,20.6,34.8,33.2,67.5,47.3,29.6,10.6,9.0,55.1,52.3,29.6,29.5,67.8,80.958,1.776,24.205,69.2,19.260,0.605,3.741,0.67750,6.69600,1.338710,0.095748,1.045688,0.949940,68.483190,0.359659,33.720663,21.228852
3,2012,1104,Alabama,-1.0,-1.0,105.0,88.1,16.9,0.883,0.656250,49.0,43.4,36.5,38.6,20.4,21.4,33.9,31.9,63.1,43.9,28.3,11.9,9.4,52.0,51.1,27.1,32.9,63.1,79.579,1.010,84.233,71.5,26.959,0.842,14.255,0.78075,11.43500,1.574844,0.154447,1.034203,0.879756,63.968330,0.226809,37.483615,25.326671
4,2012,1105,Alabama A&M,-1.0,-1.0,88.4,111.1,-22.7,0.067,0.192308,45.1,49.2,35.5,52.9,23.9,20.6,30.4,33.6,67.6,48.0,35.1,10.3,11.3,48.9,45.9,30.9,25.4,67.9,79.249,1.561,2.167,65.2,8.917,0.128,-16.046,0.10350,-18.40925,-3.145351,-0.191359,0.890988,1.082347,68.157167,-0.315889,3.471657,-9.691644
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4589,2024,1476,Stonehill,-1.0,-1.0,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.164153,-0.214380,0.914280,1.128660,69.240111,-0.400817,2.053793,-11.474784
4590,2024,1477,East Texas A&M,-1.0,-1.0,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.196470,-0.161715,0.935543,1.097258,67.458552,-0.111816,12.329532,-0.371308
4591,2024,1478,Le Moyne,-1.0,-1.0,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.228214,-0.091264,1.001228,1.092492,68.446622,-0.070744,16.213913,3.887557
4592,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
df.loc[df['OS Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2012,1109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2012,1118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2012,1121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2012,1128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4545,2024,1432,Utica,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4558,2024,1445,W Salem St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4559,2024,1446,W Texas A&M,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4592,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Betting Odds

In [37]:
df_bo = pd.read_parquet('../data/preprocessed/mens_betting/betting.parquet')

df_bo

,Season,Team,Implied Champion Probability
0,2012,Kentucky,0.350877
1,2012,Kansas,0.090909
2,2012,Ohio State,0.133333
3,2012,Louisville,0.029412
4,2012,Syracuse,0.076923
...,...,...,...
750,2024,Montana State,0.000500
751,2024,South Dakota State,0.000500
752,2024,St Peter's,0.000500
753,2024,Stetson,0.000500


In [38]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_bo['Team'].unique())

df_match.head(25)

  0%|          | 0/221 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,U Mass,umass,73
1,Kentucky,kentucky,100
2,Mount St Mary's,mount st. mary's,100
3,NC Central,nc central,100
4,New Orleans,new orleans,100
5,North Dakota,north dakota,100
6,Northern Kentucky,northern kentucky,100
7,Troy,troy,100
8,UC Davis,uc davis,100
9,Loyola Chicago,loyola chicago,100


In [39]:
df_bo.insert(1, 'TeamID', df_bo['Team'].map(team_to_spelling).map(spelling_to_id))

df_bo

,Season,TeamID,Team,Implied Champion Probability
0,2012,1246,Kentucky,0.350877
1,2012,1242,Kansas,0.090909
2,2012,1326,Ohio State,0.133333
3,2012,1257,Louisville,0.029412
4,2012,1393,Syracuse,0.076923
...,...,...,...,...
750,2024,1286,Montana State,0.000500
751,2024,1355,South Dakota State,0.000500
752,2024,1389,St Peter's,0.000500
753,2024,1391,Stetson,0.000500


In [40]:
df = pd.merge(
    df,
    df_bo.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,1102,Air Force,-1.0,-1.0,98.5,100.0,-1.5,0.457,0.407407,51.1,48.4,39.4,38.2,20.6,21.5,19.8,31.5,62.7,48.4,32.3,10.0,9.4,64.0,60.6,40.0,41.6,62.3,79.876,1.607,0.766,68.2,21.142,0.578,2.881,0.46275,-1.37775,0.155360,-0.020097,0.979952,1.000049,62.836286,-0.066193,24.057973,10.314879,NaN
2,2012,1103,Akron,0.0,-0.5,105.1,96.9,8.2,0.718,0.636364,51.5,46.4,40.0,34.0,21.0,20.6,34.8,33.2,67.5,47.3,29.6,10.6,9.0,55.1,52.3,29.6,29.5,67.8,80.958,1.776,24.205,69.2,19.260,0.605,3.741,0.67750,6.69600,1.338710,0.095748,1.045688,0.949940,68.483190,0.359659,33.720663,21.228852,NaN
3,2012,1104,Alabama,-1.0,-1.0,105.0,88.1,16.9,0.883,0.656250,49.0,43.4,36.5,38.6,20.4,21.4,33.9,31.9,63.1,43.9,28.3,11.9,9.4,52.0,51.1,27.1,32.9,63.1,79.579,1.010,84.233,71.5,26.959,0.842,14.255,0.78075,11.43500,1.574844,0.154447,1.034203,0.879756,63.968330,0.226809,37.483615,25.326671,0.006623
4,2012,1105,Alabama A&M,-1.0,-1.0,88.4,111.1,-22.7,0.067,0.192308,45.1,49.2,35.5,52.9,23.9,20.6,30.4,33.6,67.6,48.0,35.1,10.3,11.3,48.9,45.9,30.9,25.4,67.9,79.249,1.561,2.167,65.2,8.917,0.128,-16.046,0.10350,-18.40925,-3.145351,-0.191359,0.890988,1.082347,68.157167,-0.315889,3.471657,-9.691644,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4589,2024,1476,Stonehill,-1.0,-1.0,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.164153,-0.214380,0.914280,1.128660,69.240111,-0.400817,2.053793,-11.474784,NaN
4590,2024,1477,East Texas A&M,-1.0,-1.0,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.196470,-0.161715,0.935543,1.097258,67.458552,-0.111816,12.329532,-0.371308,NaN
4591,2024,1478,Le Moyne,-1.0,-1.0,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.228214,-0.091264,1.001228,1.092492,68.446622,-0.070744,16.213913,3.887557,NaN
4592,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
df.loc[df['Implied Champion Probability'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability
0,2012,1101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,1102,Air Force,-1.0,-1.0,98.5,100.0,-1.5,0.457,0.407407,51.1,48.4,39.4,38.2,20.6,21.5,19.8,31.5,62.7,48.4,32.3,10.0,9.4,64.0,60.6,40.0,41.6,62.3,79.876,1.607,0.766,68.2,21.142,0.578,2.881,0.46275,-1.37775,0.155360,-0.020097,0.979952,1.000049,62.836286,-0.066193,24.057973,10.314879,NaN
2,2012,1103,Akron,0.0,-0.5,105.1,96.9,8.2,0.718,0.636364,51.5,46.4,40.0,34.0,21.0,20.6,34.8,33.2,67.5,47.3,29.6,10.6,9.0,55.1,52.3,29.6,29.5,67.8,80.958,1.776,24.205,69.2,19.260,0.605,3.741,0.67750,6.69600,1.338710,0.095748,1.045688,0.949940,68.483190,0.359659,33.720663,21.228852,NaN
4,2012,1105,Alabama A&M,-1.0,-1.0,88.4,111.1,-22.7,0.067,0.192308,45.1,49.2,35.5,52.9,23.9,20.6,30.4,33.6,67.6,48.0,35.1,10.3,11.3,48.9,45.9,30.9,25.4,67.9,79.249,1.561,2.167,65.2,8.917,0.128,-16.046,0.10350,-18.40925,-3.145351,-0.191359,0.890988,1.082347,68.157167,-0.315889,3.471657,-9.691644,NaN
5,2012,1106,Alabama St,0.0,-0.5,86.2,104.3,-18.1,0.101,0.344828,42.7,48.1,34.1,41.7,22.6,23.6,32.6,33.1,64.9,49.8,29.4,8.4,12.0,54.4,58.2,36.3,29.6,64.8,77.282,2.267,0.200,57.5,11.403,0.164,-14.064,0.27650,-8.94500,-2.145172,-0.151396,0.863822,1.015218,65.426361,-0.181268,9.408010,-3.481456,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4589,2024,1476,Stonehill,-1.0,-1.0,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.164153,-0.214380,0.914280,1.128660,69.240111,-0.400817,2.053793,-11.474784,NaN
4590,2024,1477,East Texas A&M,-1.0,-1.0,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.196470,-0.161715,0.935543,1.097258,67.458552,-0.111816,12.329532,-0.371308,NaN
4591,2024,1478,Le Moyne,-1.0,-1.0,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.228214,-0.091264,1.001228,1.092492,68.446622,-0.070744,16.213913,3.887557,NaN
4592,2024,1479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Map to Matchups

In [42]:
df_mod = pd.read_csv(r'..\data\unprocessed\kaggle\MNCAATourneyDetailedResults.csv')[['Season', 'DayNum', 'WTeamID', 'LTeamID']]

df_mod = df_mod.loc[df_mod['Season'].between(2012, SEASON, inclusive='left'), :].reset_index(drop=True)

# fix 2021 dates
df_mod.loc[
    (df_mod['Season'] == 2021) & 
    (df_mod['DayNum'] < 140), 
    'DayNum'
] = df_mod.loc[
    (df_mod['Season'] == 2021) & 
    (df_mod['DayNum'] < 140), 
    'DayNum'
] - 1

df_mod.loc[
    (df_mod['Season'] == 2021) & 
    (df_mod['DayNum'].between(140, 150)), 
    'DayNum'
] = df_mod.loc[
    (df_mod['Season'] == 2021) & 
    (df_mod['DayNum'].between(140, 150)), 
    'DayNum'
] - 2

# get rid of play-in games
df_mod = df_mod.loc[df_mod['DayNum'] > 135, :].reset_index(drop=True)

df_mod

,Season,DayNum,WTeamID,LTeamID
0,2012,136,1124,1355
1,2012,136,1160,1424
2,2012,136,1211,1452
3,2012,136,1231,1308
4,2012,136,1235,1163
...,...,...,...,...
750,2024,146,1301,1181
751,2024,146,1345,1397
752,2024,152,1163,1104
753,2024,152,1345,1301


Get round of each game

In [43]:
df_mod['Round'] = 1
df_mod.loc[df_mod['DayNum'].between(138, 139), 'Round'] = 2
df_mod.loc[df_mod['DayNum'].between(140, 144), 'Round'] = 3
df_mod.loc[df_mod['DayNum'].between(145, 149), 'Round'] = 4
df_mod.loc[df_mod['DayNum'].between(150, 153), 'Round'] = 5
df_mod.loc[df_mod['DayNum'] == 154, 'Round'] = 6

df_mod

,Season,DayNum,WTeamID,LTeamID,Round
0,2012,136,1124,1355,1
1,2012,136,1160,1424,1
2,2012,136,1211,1452,1
3,2012,136,1231,1308,1
4,2012,136,1235,1163,1
...,...,...,...,...,...
750,2024,146,1301,1181,4
751,2024,146,1345,1397,4
752,2024,152,1163,1104,5
753,2024,152,1345,1301,5


Check if there are any irregularities of number of games in a round

In [44]:
for season in range(2012, SEASON):
    if season != 2020:  # cancelled
        for round_ in range(1, 7):
            if df_mod.loc[(df_mod['Season'] == season) & (df_mod['Round'] == round_), :].shape[0] != 2**(6 - round_):
                print(f"{season}, {round_} : {df_mod.loc[(df_mod['Season'] == season) & (df_mod['Round'] == round_), :].shape[0]}")

2021, 1 : 31


Remap to Team A / Team B format

In [45]:
df_mod = pd.DataFrame({
    'Season': list(df_mod['Season'])*2,
    'Round': list(df_mod['Round'])*2,
    'Result': [1 for _ in range(df_mod.shape[0])] + [-1 for _ in range(df_mod.shape[0])],
    'Team A ID': list(df_mod['WTeamID']) + list(df_mod['LTeamID']),
    'Team B ID': list(df_mod['LTeamID']) + list(df_mod['WTeamID']),
})

df_mod

,Season,Round,Result,Team A ID,Team B ID
0,2012,1,1,1124,1355
1,2012,1,1,1160,1424
2,2012,1,1,1211,1452
3,2012,1,1,1231,1308
4,2012,1,1,1235,1163
...,...,...,...,...,...
1505,2024,4,-1,1181,1301
1506,2024,4,-1,1397,1345
1507,2024,5,-1,1104,1163
1508,2024,5,-1,1301,1345


Get Head-to-Head

In [46]:
df_h2h = pd.read_parquet('../data/preprocessed/mens_h2h/h2h.parquet')

df_h2h

,Season,Team A,Team B,Head to Head,Common Opps
0,2012,Air Force,Akron,NaN,-0.146509
1,2012,Air Force,Alabama A&M,NaN,0.781304
2,2012,Air Force,Alabama State,NaN,0.848025
3,2012,Air Force,Alcorn State,NaN,0.723713
4,2012,Air Force,American,NaN,0.588376
...,...,...,...,...,...
835373,2024,Youngstown State,William & Mary,NaN,1.438673
835374,2024,Youngstown State,Winthrop,NaN,0.006061
835375,2024,Youngstown State,Wisconsin,NaN,-0.111978
835376,2024,Youngstown State,Wright State,0.791622,-0.187724


In [47]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_h2h['Team A'].unique())

df_match.head(25)

  0%|          | 0/365 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Hartford Hawks,hartford,73
1,St. Francis (NY) Terriers,st francis (ny),74
2,Savannah State Tigers,savannah state,80
3,Texas A&M-Commerce,tx a&m commerce,91
4,Southern California,southern california,100
5,Sam Houston,sam houston,100
6,Saint Peter's,saint peter's,100
7,Saint Mary's (CA),saint mary's (ca),100
8,Saint Louis,saint louis,100
9,Saint Joseph's,saint joseph's,100


In [48]:
df_h2h.insert(df_h2h.columns.get_loc('Team A'), 'Team A ID', df_h2h['Team A'].map(team_to_spelling).map(spelling_to_id))

df_h2h.insert(df_h2h.columns.get_loc('Team B'), 'Team B ID', df_h2h['Team B'].map(team_to_spelling).map(spelling_to_id))

df_h2h

,Season,Team A ID,Team A,Team B ID,Team B,Head to Head,Common Opps
0,2012,1102,Air Force,1103,Akron,NaN,-0.146509
1,2012,1102,Air Force,1105,Alabama A&M,NaN,0.781304
2,2012,1102,Air Force,1106,Alabama State,NaN,0.848025
3,2012,1102,Air Force,1108,Alcorn State,NaN,0.723713
4,2012,1102,Air Force,1110,American,NaN,0.588376
...,...,...,...,...,...,...,...
835373,2024,1464,Youngstown State,1456,William & Mary,NaN,1.438673
835374,2024,1464,Youngstown State,1457,Winthrop,NaN,0.006061
835375,2024,1464,Youngstown State,1458,Wisconsin,NaN,-0.111978
835376,2024,1464,Youngstown State,1460,Wright State,0.791622,-0.187724


In [49]:
df_mod = pd.merge(
    df_mod,
    df_h2h[['Season', 'Team A ID', 'Team B ID', 'Head to Head', 'Common Opps']],
    how='left',
    on=['Season', 'Team A ID', 'Team B ID'],
)

df_mod

,Season,Round,Result,Team A ID,Team B ID,Head to Head,Common Opps
0,2012,1,1,1124,1355,NaN,NaN
1,2012,1,1,1160,1424,NaN,-0.354522
2,2012,1,1,1211,1452,NaN,0.812990
3,2012,1,1,1231,1308,NaN,NaN
4,2012,1,1,1235,1163,NaN,1.186719
...,...,...,...,...,...,...,...
1505,2024,4,-1,1181,1301,0.092813,0.427451
1506,2024,4,-1,1397,1345,-0.571735,0.206760
1507,2024,5,-1,1104,1163,NaN,-0.512516
1508,2024,5,-1,1301,1345,NaN,-1.260545


Get team names

In [50]:
df_teams = pd.read_csv(r'..\data\unprocessed\kaggle\MTeams.csv')

df_teams

,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2025
1,1102,Air Force,1985,2025
2,1103,Akron,1985,2025
3,1104,Alabama,1985,2025
4,1105,Alabama A&M,2000,2025
...,...,...,...,...
375,1476,Stonehill,2023,2025
376,1477,East Texas A&M,2023,2025
377,1478,Le Moyne,2024,2025
378,1479,Mercyhurst,2025,2025


In [51]:
id_to_team = dict(zip(df_teams['TeamID'], df_teams['TeamName']))

df_mod.insert(df_mod.columns.get_loc('Team A ID') + 1, 'Team A', df_mod['Team A ID'].map(id_to_team))
df_mod.insert(df_mod.columns.get_loc('Team B ID') + 1, 'Team B', df_mod['Team B ID'].map(id_to_team))

df_mod

,Season,Round,Result,Team A ID,Team A,Team B ID,Team B,Head to Head,Common Opps
0,2012,1,1,1124,Baylor,1355,S Dakota St,NaN,NaN
1,2012,1,1,1160,Colorado,1424,UNLV,NaN,-0.354522
2,2012,1,1,1211,Gonzaga,1452,West Virginia,NaN,0.812990
3,2012,1,1,1231,Indiana,1308,New Mexico St,NaN,NaN
4,2012,1,1,1235,Iowa St,1163,Connecticut,NaN,1.186719
...,...,...,...,...,...,...,...,...,...
1505,2024,4,-1,1181,Duke,1301,NC State,0.092813,0.427451
1506,2024,4,-1,1397,Tennessee,1345,Purdue,-0.571735,0.206760
1507,2024,5,-1,1104,Alabama,1163,Connecticut,NaN,-0.512516
1508,2024,5,-1,1301,NC State,1345,Purdue,NaN,-1.260545


Map features

In [52]:
team_a_features = pd.merge(
    df_mod[['Season', 'Team A ID']],
    df.drop(columns=['Team']),
    how='left',
    left_on=['Season', 'Team A ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team A ID', 'TeamID'])

team_b_features = pd.merge(
    df_mod[['Season', 'Team B ID']],
    df.drop(columns=['Team']),
    how='left',
    left_on=['Season', 'Team B ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team B ID', 'TeamID'])

df_features = team_a_features - team_b_features

df_features['Team A ADJ OE Team B ADJ DE'] = team_a_features['ADJ OE'] + team_b_features['ADJ DE']
df_features['Team B ADJ OE Team A ADJ DE'] = team_b_features['ADJ OE'] + team_a_features['ADJ DE']

df_features['Team A Offense Team B Defense'] = team_a_features['Adjusted Offense'] + team_b_features['Adjusted Defense']
df_features['Team B Offense Team A Defense'] = team_b_features['Adjusted Offense'] + team_a_features['Adjusted Defense']

df_features['Team A BARTHAG'] = team_a_features['BARTHAG']
df_features['Team B BARTHAG'] = team_b_features['BARTHAG']

df_features

,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability,Team A ADJ OE Team B ADJ DE,Team B ADJ OE Team A ADJ DE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A BARTHAG,Team B BARTHAG
0,0.0,1.250000,2.7,-7.1,9.8,0.143,0.006629,-1.2,-2.7,-3.4,2.2,5.0,2.1,7.1,4.7,1.2,-5.0,1.5,7.4,0.9,2.9,6.0,-5.2,1.5,0.4,2.572,0.092,54.372,2.2,18.008,0.070,2.416,0.42825,20.00450,2.338760,0.067615,0.002898,-0.064718,0.583440,0.117964,6.155373,6.307358,NaN,215.1,205.3,2.102757,2.035141,0.910,0.767
1,-1.0,-1.000000,-5.1,1.9,-7.0,-0.114,-0.093750,-3.9,0.9,7.2,-0.9,-0.3,-2.3,-3.3,-1.7,-3.7,1.4,-0.2,-3.6,3.9,-15.6,-1.2,-6.9,1.5,-4.0,-0.808,-0.550,-17.999,0.3,-1.350,-0.085,-4.570,-0.16850,-8.01825,-1.031018,-0.086742,-0.061467,0.025275,-3.751519,-0.226425,-5.594902,-5.063750,-0.006579,196.5,203.5,1.942409,2.029151,0.761,0.875
2,0.0,-0.750000,-0.3,-1.2,0.9,0.014,0.212702,4.7,-2.8,8.9,-4.7,0.6,-0.7,-6.4,-2.5,1.8,-3.7,-0.5,2.0,-1.4,-1.7,-2.5,2.0,3.4,0.7,0.694,-0.074,-24.821,4.4,-10.613,-0.028,-3.011,-0.02675,-2.83100,0.274774,0.029706,0.004889,-0.024816,0.670748,0.164332,6.373943,5.266265,0.001314,206.1,205.2,2.033816,2.004111,0.862,0.848
3,0.0,0.000000,13.0,-0.9,13.9,0.190,0.030303,4.0,-0.1,-6.2,1.0,-1.6,-0.1,-5.2,1.5,-3.8,-2.2,2.9,-1.4,2.6,-0.7,-8.4,3.1,0.1,-3.3,0.873,-0.372,59.973,9.8,17.567,0.189,7.508,0.00800,1.45825,1.508085,0.104455,0.097244,-0.007210,-3.018990,0.115812,6.609197,6.794695,NaN,216.5,202.6,2.116803,2.012348,0.918,0.728
4,-7.0,-3.250000,0.9,1.0,-0.1,-0.003,0.081439,2.7,3.9,4.3,0.8,-0.8,0.7,-4.6,-8.3,3.4,9.4,-4.5,-10.7,3.1,5.1,-0.6,13.0,-3.5,3.0,-1.707,1.082,-3.734,1.8,-2.296,-0.193,-13.367,-0.20000,-13.53850,0.155641,0.007995,0.021385,0.013390,2.322124,-0.035981,3.194011,3.145467,-0.009950,205.5,205.6,2.038216,2.030221,0.855,0.858
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1505,1.0,2.000000,7.1,-3.8,10.9,0.126,0.138889,4.5,-1.5,1.3,-4.3,0.6,-1.0,3.2,-3.4,-1.1,-0.4,-2.3,0.5,4.1,7.7,2.3,3.9,0.2,-1.0,1.687,-1.262,43.185,-1.2,-1.832,0.080,5.906,0.13675,10.16675,0.102572,0.121062,0.073952,-0.047109,-0.912177,0.151976,4.388026,3.498631,0.027283,221.4,210.5,2.206684,2.085623,0.926,0.800
1506,2.0,0.333333,-10.7,-3.2,-7.5,-0.028,-0.128788,-4.5,-2.3,-8.5,12.8,-1.9,4.9,-5.0,3.9,1.6,-3.8,0.0,4.0,2.0,-3.3,-5.6,6.3,4.2,1.6,-3.043,0.405,2.815,2.8,-1.438,0.007,-0.478,-0.01675,-1.74175,-0.279607,-0.035856,-0.077972,-0.042117,1.337846,-0.133075,-4.770799,-3.938249,-0.070833,209.7,217.2,2.099227,2.135083,0.938,0.966
1507,-4.0,-0.666667,-1.2,7.3,-8.5,-0.048,-0.255515,-0.8,4.8,1.9,7.1,1.1,-0.6,-1.6,3.1,8.0,7.4,0.0,-4.0,4.2,-12.2,-0.9,5.9,3.5,7.9,0.817,0.392,-30.485,4.2,2.281,-0.009,-3.217,-0.00150,-0.50050,-0.392997,-0.059663,0.001401,0.061064,7.406847,-0.265038,-10.958855,-10.211180,-0.197832,219.7,228.2,2.178061,2.237724,0.921,0.969
1508,0.0,-1.333333,-12.5,6.4,-18.9,-0.166,-0.267677,-5.3,2.8,-10.3,8.9,-2.8,3.9,-8.8,4.6,-0.1,1.4,3.5,-0.3,1.7,-18.4,-6.7,-1.5,-1.9,0.3,-2.671,0.579,-3.902,1.3,-5.629,-0.106,-9.694,-0.13000,-9.16325,-1.249299,-0.166874,-0.105671,0.061203,0.461882,-0.364008,-11.834769,-10.292074,-0.128358,207.9,226.8,2.071529,2.238403,0.800,0.966


In [53]:
df_mod[df_features.columns] = df_features

df_mod

,Season,Round,Result,Team A ID,Team A,Team B ID,Team B,Head to Head,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability,Team A ADJ OE Team B ADJ DE,Team B ADJ OE Team A ADJ DE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A BARTHAG,Team B BARTHAG
0,2012,1,1,1124,Baylor,1355,S Dakota St,NaN,NaN,0.0,1.250000,2.7,-7.1,9.8,0.143,0.006629,-1.2,-2.7,-3.4,2.2,5.0,2.1,7.1,4.7,1.2,-5.0,1.5,7.4,0.9,2.9,6.0,-5.2,1.5,0.4,2.572,0.092,54.372,2.2,18.008,0.070,2.416,0.42825,20.00450,2.338760,0.067615,0.002898,-0.064718,0.583440,0.117964,6.155373,6.307358,NaN,215.1,205.3,2.102757,2.035141,0.910,0.767
1,2012,1,1,1160,Colorado,1424,UNLV,NaN,-0.354522,-1.0,-1.000000,-5.1,1.9,-7.0,-0.114,-0.093750,-3.9,0.9,7.2,-0.9,-0.3,-2.3,-3.3,-1.7,-3.7,1.4,-0.2,-3.6,3.9,-15.6,-1.2,-6.9,1.5,-4.0,-0.808,-0.550,-17.999,0.3,-1.350,-0.085,-4.570,-0.16850,-8.01825,-1.031018,-0.086742,-0.061467,0.025275,-3.751519,-0.226425,-5.594902,-5.063750,-0.006579,196.5,203.5,1.942409,2.029151,0.761,0.875
2,2012,1,1,1211,Gonzaga,1452,West Virginia,NaN,0.812990,0.0,-0.750000,-0.3,-1.2,0.9,0.014,0.212702,4.7,-2.8,8.9,-4.7,0.6,-0.7,-6.4,-2.5,1.8,-3.7,-0.5,2.0,-1.4,-1.7,-2.5,2.0,3.4,0.7,0.694,-0.074,-24.821,4.4,-10.613,-0.028,-3.011,-0.02675,-2.83100,0.274774,0.029706,0.004889,-0.024816,0.670748,0.164332,6.373943,5.266265,0.001314,206.1,205.2,2.033816,2.004111,0.862,0.848
3,2012,1,1,1231,Indiana,1308,New Mexico St,NaN,NaN,0.0,0.000000,13.0,-0.9,13.9,0.190,0.030303,4.0,-0.1,-6.2,1.0,-1.6,-0.1,-5.2,1.5,-3.8,-2.2,2.9,-1.4,2.6,-0.7,-8.4,3.1,0.1,-3.3,0.873,-0.372,59.973,9.8,17.567,0.189,7.508,0.00800,1.45825,1.508085,0.104455,0.097244,-0.007210,-3.018990,0.115812,6.609197,6.794695,NaN,216.5,202.6,2.116803,2.012348,0.918,0.728
4,2012,1,1,1235,Iowa St,1163,Connecticut,NaN,1.186719,-7.0,-3.250000,0.9,1.0,-0.1,-0.003,0.081439,2.7,3.9,4.3,0.8,-0.8,0.7,-4.6,-8.3,3.4,9.4,-4.5,-10.7,3.1,5.1,-0.6,13.0,-3.5,3.0,-1.707,1.082,-3.734,1.8,-2.296,-0.193,-13.367,-0.20000,-13.53850,0.155641,0.007995,0.021385,0.013390,2.322124,-0.035981,3.194011,3.145467,-0.009950,205.5,205.6,2.038216,2.030221,0.855,0.858
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1505,2024,4,-1,1181,Duke,1301,NC State,0.092813,0.427451,1.0,2.000000,7.1,-3.8,10.9,0.126,0.138889,4.5,-1.5,1.3,-4.3,0.6,-1.0,3.2,-3.4,-1.1,-0.4,-2.3,0.5,4.1,7.7,2.3,3.9,0.2,-1.0,1.687,-1.262,43.185,-1.2,-1.832,0.080,5.906,0.13675,10.16675,0.102572,0.121062,0.073952,-0.047109,-0.912177,0.151976,4.388026,3.498631,0.027283,221.4,210.5,2.206684,2.085623,0.926,0.800
1506,2024,4,-1,1397,Tennessee,1345,Purdue,-0.571735,0.206760,2.0,0.333333,-10.7,-3.2,-7.5,-0.028,-0.128788,-4.5,-2.3,-8.5,12.8,-1.9,4.9,-5.0,3.9,1.6,-3.8,0.0,4.0,2.0,-3.3,-5.6,6.3,4.2,1.6,-3.043,0.405,2.815,2.8,-1.438,0.007,-0.478,-0.01675,-1.74175,-0.279607,-0.035856,-0.077972,-0.042117,1.337846,-0.133075,-4.770799,-3.938249,-0.070833,209.7,217.2,2.099227,2.135083,0.938,0.966
1507,2024,5,-1,1104,Alabama,1163,Connecticut,NaN,-0.512516,-4.0,-0.666667,-1.2,7.3,-8.5,-0.048,-0.255515,-0.8,4.8,1.9,7.1,1.1,-0.6,-1.6,3.1,8.0,7.4,0.0,-4.0,4.2,-12.2,-0.9,5.9,3.5,7.9,0.817,0.392,-30.485,4.2,2.281,-0.009,-3.217,-0.00150,-0.50050,-0.392997,-0.059663,0.001401,0.061064,7.406847,-0.265038,-10.958855,-10.211180,-0.197832,219.7,228.2,2.178061,2.237724,0.921,0.969
1508,2024,5,-1,1301,NC State,1345,Purdue,NaN,-1.260545,0.0,-1.333333,-12.5,6.4,-18.9,-0.166,-0.267677,-5.3,2.8,-10.3,8.9,-2.8,3.9,-8.8,4

In [54]:
df_mod.to_parquet('../data/preprocessed/mens_model_data/model_data.parquet')

'Done'

'Done'